# Introduction

This notebook demonstrates how to train custom openWakeWord models using pre-defined datasets and an automated process for dataset generation and training. While not guaranteed to always produce the best performing model, the methods shown in this notebook often produce baseline models with releatively strong performance.

Manual data preparation and model training (e.g., see the [training models](training_models.ipynb) notebook) remains an option for when full control over the model development process is needed.

At a high level, the automatic training process takes advantages of several techniques to try and produce a good model, including:

- Early-stopping and checkpoint averaging (similar to [stochastic weight averaging](https://arxiv.org/abs/1803.05407)) to search for the best models found during training, according to the validation data
- Variable learning rates with cosine decay and multiple cycles
- Adaptive batch construction to focus on only high-loss examples when the model begins to converge, combined with gradient accumulation to ensure that batch sizes are still large enough for stable training
- Cycical weight schedules for negative examples to help the model reduce false-positive rates

See the contents of the `train.py` file for more details.

# Environment Setup

To begin, we'll need to install the requirements for training custom models. In particular, a relatively recent version of Pytorch and custom fork of the [piper-sample-generator](https://github.com/dscripka/piper-sample-generator) library for generating synthetic examples for the custom model.

**Important Note!** Currently, automated model training is only supported on linux systems due to the requirements of the text to speech library used for synthetic sample generation (Piper). It may be possible to use Piper on Windows/Mac systems, but that has not (yet) been tested.

In [4]:
# 1. Install system dependencies
!apt-get update -qq && apt-get install -y -qq build-essential python3-dev ffmpeg libsndfile1

# 2. Clone piper-sample-generator and pin to the exact v2 commit openWakeWord expects
!rm -rf piper-sample-generator
!git clone https://github.com/rhasspy/piper-sample-generator
!git -C piper-sample-generator checkout f1988a4
!mkdir -p piper-sample-generator/models
!wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'

# 3. Install Python 3.12 compatible packages
!pip install -q piper-phonemize-cross webrtcvad mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0
!pip install -q speechbrain==0.5.14 audiomentations==0.33.0 torch-audiomentations==0.11.0 acoustics==0.2.6
!pip install -q pronouncing==0.2.0 deep-phonemizer==0.0.19 onnx onnxruntime soundfile librosa datasets scipy tqdm pyyaml

# 4. Clone and install openWakeWord (--no-deps avoids the broken tflite-runtime wheel on Python 3.12)
!rm -rf openwakeword
!git clone https://github.com/dscripka/openwakeword
!pip install -q -e ./openwakeword --no-deps

# 5. Download required openWakeWord backbone models
import os
os.makedirs("./openwakeword/openwakeword/resources/models", exist_ok=True)
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx -O ./openwakeword/openwakeword/resources/models/embedding_model.onnx
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite -O ./openwakeword/openwakeword/resources/models/embedding_model.tflite
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx -O ./openwakeword/openwakeword/resources/models/melspectrogram.onnx
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite -O ./openwakeword/openwakeword/resources/models/melspectrogram.tflite

# 6. Patch torchaudio 2.5+ removals & broken TFLite conversion in openWakeWord and torch_audiomentations
import glob

# Patch torch_audiomentations set_audio_backend call
for p in glob.glob("/usr/local/lib/python*/dist-packages/torch_audiomentations/utils/io.py"):
    with open(p, "r") as f:
        code = f.read()
    code = code.replace('torchaudio.set_audio_backend("soundfile")', 'pass')
    with open(p, "w") as f:
        f.write(code)

# Patch openwakeword/data.py to use soundfile instead of removed torchaudio.info / set_audio_backend
with open("openwakeword/openwakeword/data.py", "r") as f:
    data_code = f.read()
data_code = data_code.replace('torchaudio.set_audio_backend("soundfile")', 'pass')
# Inject a torchaudio.info shim using soundfile at the top of data.py
shim = """
import soundfile as _sf
import torchaudio as _ta
if not hasattr(_ta, 'info'):
    class _AudioMeta:
        def __init__(self, path):
            info = _sf.info(path)
            self.sample_rate = info.samplerate
            self.num_frames = info.frames
            self.num_channels = info.channels
    _ta.info = _AudioMeta
"""
data_code = shim + "\n" + data_code
with open("openwakeword/openwakeword/data.py", "w") as f:
    f.write(data_code)

# Patch openwakeword/train.py to skip broken onnx_tf TFLite conversion (retaining clean .onnx output)
with open("openwakeword/openwakeword/train.py", "r") as f:
    train_code = f.read()
train_code = train_code.replace(
    'convert_onnx_to_tflite(os.path.join(config["output_dir"], config["model_name"] + ".onnx"),',
    '# Skipped TFLite conversion (broken on Py3.12)\n    # convert_onnx_to_tflite(os.path.join(config["output_dir"], config["model_name"] + ".onnx"),'
)
train_code = train_code.replace(
    'os.path.join(config["output_dir"], config["model_name"] + ".tflite"))',
    '# os.path.join(config["output_dir"], config["model_name"] + ".tflite"))'
)
with open("openwakeword/openwakeword/train.py", "w") as f:
    f.write(train_code)

print("✅ Environment setup and patches applied successfully!")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package javascript-common.
(Reading database ... 122807 files and directories currently installed.)
Preparing to unpack .../javascript-common_11+nmu1_all.deb ...
Unpacking javascript-common (11+nmu1) ...
Selecting previously unselected package libjs-underscore.
Preparing to unpack .../libjs-underscore_1.13.4~dfsg+~1.11.4-3_all.deb ...
Unpacking libjs-underscore (1.13.4~dfsg+~1.11.4-3) ...
Selecting previously unselected package libjs-sphinxdoc.
Preparing to unpack .../libjs-sphinxdoc_7.2.6-6_all.deb ...
Unpacking libjs-sphinxdoc (7.2.6-6) ...
Selecting previously unselected package python3.12-dev.
Preparing to unpack .../python3.12-dev_3.12.3-1ubuntu0.17_amd64.deb ...
Unpacking python3.12-dev (3.12.3-1ubuntu0.17) ...
Selecting previously unselected package python3-dev.

In [5]:
import os
import io
import sys
import uuid
import yaml
import glob
from pathlib import Path
import numpy as np
import torch
import scipy.io.wavfile
import soundfile as sf
import librosa
import datasets
from tqdm import tqdm

# Ensure openwakeword and piper-sample-generator are on sys.path
if os.path.abspath("./openwakeword") not in sys.path:
    sys.path.insert(0, os.path.abspath("./openwakeword"))
if os.path.abspath("./piper-sample-generator") not in sys.path:
    sys.path.insert(0, os.path.abspath("./piper-sample-generator"))

# Download Data

When training new openWakeWord models using the automated procedure, four specific types of data are required:

1) Synthetic examples of the target word/phrase generated with text-to-speech models

2) Synthetic examples of adversarial words/phrases generated with text-to-speech models

3) Room impulse reponses and noise/background audio data to augment the synthetic examples and make them more realistic

4) Generic "negative" audio data that is very unlikely to contain examples of the target word/phrase in the context where the model should detect it. This data can be the original audio data, or precomputed openWakeWord features ready for model training.

5) Validation data to use for early-stopping when training the model.

For the purposes of this notebook, all five of these sources will either be generated manually or can be obtained from HuggingFace thanks to their excellent `datasets` library and extremely generous hosting policy. Also note that while only a portion of some datasets are downloaded, for the best possible performance it is recommended to download the entire dataset and keep a local copy for future training runs.

In [6]:
output_dir = "./mit_rirs"
os.makedirs(output_dir, exist_ok=True)

if len(glob.glob(f"{output_dir}/*.wav")) == 0:
    rir_dataset = datasets.load_dataset(
        "davidscripka/MIT_environmental_impulse_responses",
        split="train",
        streaming=True
    ).cast_column("audio", datasets.Audio(decode=False))

    for row in tqdm(rir_dataset, desc="Downloading MIT RIRs"):
        audio_info = row["audio"]
        name = Path(audio_info["path"]).name
        temp_path = os.path.join(output_dir, f"temp_{name}")

        # Write bytes to disk first to bypass BytesIO format errors
        with open(temp_path, "wb") as f:
            f.write(audio_info["bytes"])

        # Load from disk (handles format properly) and save to 16kHz WAV
        audio_arr, _ = librosa.load(temp_path, sr=16000, mono=True)
        scipy.io.wavfile.write(
            os.path.join(output_dir, name),
            16000,
            (audio_arr * 32767).astype(np.int16)
        )
        os.remove(temp_path)  # Cleanup temp file

print(f"✅ MIT RIR files ready: {len(glob.glob(f'{output_dir}/*.wav'))} clips")

Resolving data files:   0%|          | 0/270 [00:00<?, ?it/s]

LibsndfileError: Error opening <_io.BytesIO object at 0x784761d13ba0>: Format not recognised.

In [ ]:
## 1. Audioset Dataset
os.makedirs("audioset", exist_ok=True)
fname = "bal_train09.tar"
out_dir = f"audioset/{fname}"
if not os.path.exists(out_dir):
    link = "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/" + fname
    !wget -q --show-progress -O {out_dir} {link}
    !cd audioset && tar -xf bal_train09.tar

output_dir_audioset = "./audioset_16k"
os.makedirs(output_dir_audioset, exist_ok=True)

flac_files = list(Path("audioset/audio").glob("**/*.flac"))
if len(glob.glob(f"{output_dir_audioset}/*.wav")) < len(flac_files):
    for flac_path in tqdm(flac_files, desc="Converting AudioSet to 16kHz"):
        name = flac_path.name.replace(".flac", ".wav")
        try:
            audio_arr, _ = librosa.load(str(flac_path), sr=16000, mono=True)
            scipy.io.wavfile.write(
                os.path.join(output_dir_audioset, name),
                16000,
                (audio_arr * 32767).astype(np.int16)
            )
        except Exception:
            pass  # Skip corrupted flacs

## 2. Free Music Archive (FMA) Dataset
output_dir_fma = "./fma"
os.makedirs(output_dir_fma, exist_ok=True)
n_hours = 1
target_clips = n_hours * 3600 // 30

if len(glob.glob(f"{output_dir_fma}/*.wav")) < target_clips:
    fma_dataset = datasets.load_dataset(
        "rudraml/fma",
        name="small",
        split="train",
        streaming=True
    ).cast_column("audio", datasets.Audio(decode=False))
    fma_iter = iter(fma_dataset)

    for _ in tqdm(range(target_clips), desc="Downloading FMA"):
        mp3_path = None
        try:
            row = next(fma_iter)
            audio_info = row["audio"]
            name = Path(audio_info["path"]).name
            mp3_path = os.path.join(output_dir_fma, name)
            wav_name = name.replace(".mp3", ".wav")
            wav_path = os.path.join(output_dir_fma, wav_name)

            # Write MP3 bytes to disk first so librosa can use ffmpeg decoding
            with open(mp3_path, "wb") as f:
                f.write(audio_info["bytes"])

            audio_arr, _ = librosa.load(mp3_path, sr=16000, mono=True)
            scipy.io.wavfile.write(
                wav_path,
                16000,
                (audio_arr * 32767).astype(np.int16)
            )
            os.remove(mp3_path) # Cleanup

        except StopIteration:
            break
        except Exception:
            # Cleanup if it fails mid-process
            if mp3_path and os.path.exists(mp3_path):
                os.remove(mp3_path)
            continue

print(f"✅ Background audio ready: {len(glob.glob('./audioset_16k/*.wav'))} AudioSet + {len(glob.glob('./fma/*.wav'))} FMA clips")

In [ ]:
# Download pre-computed openWakeWord features for training and validation

# training set (~2,000 hours from the ACAV100M Dataset)
# See https://huggingface.co/datasets/davidscripka/openwakeword_features for more information
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy

# validation set for false positive rate estimation (~11 hours)
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

# Define Training Configuration

For automated model training openWakeWord uses a specially designed training script and a [YAML](https://yaml.org/) configuration file that defines all of the information required for training a new wake word/phrase detection model.

It is strongly recommended that you review [the example config file](../examples/custom_model.yml), as each value is fully documented there. For the purposes of this notebook, we'll read in the YAML file to modify certain configuration parameters before saving a new YAML file for training our example model. Specifically:

- We'll train a detection model for the phrase "hey sebastian"
- We'll only generate 5,000 positive and negative examples (to save on time for this example)
- We'll only generate 1,000 validation positive and negative examples for early stopping (again to save time)
- The model will only be trained for 10,000 steps (larger datasets will benefit from longer training)
- We'll reduce the target metrics to account for the small dataset size and limited training.

On the topic of target metrics, there are *not* specific guidelines about what these metrics should be in practice, and you will need to conduct testing in your target deployment environment to establish good thresholds. However, from very limited testing the default values in the config file (accuracy >= 0.7, recall >= 0.5, false-positive rate <= 0.2 per hour) seem to produce models with reasonable performance.


In [ ]:
# Load default YAML config file for training
config = yaml.load(open("openwakeword/examples/custom_model.yml", 'r').read(), yaml.Loader)

# Clean up any previous failed/mispronounced runs for Cassidy
!rm -rf ./my_custom_model/Cassidy

# Phonetic spelling for Piper TTS so it pronounces "Cassidy" properly
config["target_phrase"] = ["cas ih dee"]

# Keep the final output file named Cassidy.onnx
config["model_name"] = "Cassidy"

# Training parameters (increase n_samples to 5000 and steps to 20000 if you want higher recall)
config["n_samples"] = 1000
config["n_samples_val"] = 1000
config["steps"] = 10000
config["target_accuracy"] = 0.6
config["target_recall"] = 0.25
config["output_dir"] = "./my_custom_model"
config["max_negative_weight"] = 1500

config["background_paths"] = ['./audioset_16k', './fma']
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}

with open('my_model.yaml', 'w') as file:
    yaml.dump(config, file)

print("✅ Saved my_model.yaml for wake word:", config["model_name"])

In [ ]:
# Modify values in the config and save a new version

config["target_phrase"] = ["Cassidy"]
config["model_name"] = config["target_phrase"][0].replace(" ", "_")
config["n_samples"] = 1000
config["n_samples_val"] = 1000
config["steps"] = 10000
config["target_accuracy"] = 0.6
config["target_recall"] = 0.25

config["background_paths"] = ['./audioset_16k', './fma']  # multiple background datasets are supported
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}

with open('my_model.yaml', 'w') as file:
    documents = yaml.dump(config, file)

# Train the Model

With the data downloaded and training configuration set, we can now start training the model. We'll do this in parts to better illustrate the sequence, but you can also execute every step at once for a fully automated process.

In [ ]:
# Step 1: Generate synthetic clips
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --generate_clips

# Verify and enforce 16,000 Hz 16-bit PCM on all generated WAVs
wav_dirs = [
    f"./my_custom_model/{config['model_name']}/positive_train",
    f"./my_custom_model/{config['model_name']}/positive_test",
    f"./my_custom_model/{config['model_name']}/negative_train",
    f"./my_custom_model/{config['model_name']}/negative_test",
]

resampled_count = 0
for d in wav_dirs:
    for wav_path in glob.glob(f"{d}/*.wav"):
        info = sf.info(wav_path)
        if info.samplerate != 16000:
            audio, _ = librosa.load(wav_path, sr=16000, mono=True)
            scipy.io.wavfile.write(wav_path, 16000, (audio * 32767).astype(np.int16))
            resampled_count += 1

print(f"✅ Clips generated! (Resampled {resampled_count} clips to 16kHz)")

import IPython.display as ipd
sample_wavs = glob.glob("./my_custom_model/Cassidy/positive_train/*.wav")
ipd.Audio(sample_wavs[0])

In [ ]:
# Step 2: Augment the generated clips

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips

In [ ]:
# Step 3: Train model and export Cassidy.onnx
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model

onnx_path = f"./my_custom_model/{config['model_name']}.onnx"
if os.path.exists(onnx_path):
    print(f"🎉 Success! Model saved to {onnx_path} ({os.path.getsize(onnx_path)/1024:.1f} KB)")
    from google.colab import files
    files.download(onnx_path)
else:
    print("❌ ONNX file was not found. Check the training output above.")

In [ ]:
# Step 4 (Optional): On Google Colab, sometimes the .tflite model isn't saved correctly
# If so, run this cell to retry

# Manually save to tflite as this doesn't work right in colab
def convert_onnx_to_tflite(onnx_model_path, output_path):
    """Converts an ONNX version of an openwakeword model to the Tensorflow tflite format."""
    # imports
    import onnx
    import logging
    import tempfile
    from onnx_tf.backend import prepare
    import tensorflow as tf

    # Convert to tflite from onnx model
    onnx_model = onnx.load(onnx_model_path)
    tf_rep = prepare(onnx_model, device="CPU")
    with tempfile.TemporaryDirectory() as tmp_dir:
        tf_rep.export_graph(os.path.join(tmp_dir, "tf_model"))
        converter = tf.lite.TFLiteConverter.from_saved_model(os.path.join(tmp_dir, "tf_model"))
        tflite_model = converter.convert()

        logging.info(f"####\nSaving tflite mode to '{output_path}'")
        with open(output_path, 'wb') as f:
            f.write(tflite_model)

    return None

convert_onnx_to_tflite(f"my_custom_model/{config['model_name']}.onnx", f"my_custom_model/{config['model_name']}.tflite")


After the model finishes training, the auto training script will automatically convert it to ONNX and tflite versions, saving them as `my_custom_model/<model_name>.onnx/tflite` in the present working directory, where `<model_name>` is defined in the YAML training config file. Either version can be used as normal with `openwakeword`. I recommend testing them with the [`detect_from_microphone.py`](https://github.com/dscripka/openWakeWord/blob/main/examples/detect_from_microphone.py) example script to see how the model performs!